In [0]:
%sql
-- Dimension 1: Customers (connected via Surrogate Key customer_sk)
CREATE OR REPLACE TABLE workspace.freshmart_gold.dim_customer AS
SELECT customer_sk, customer_id, customer_name, city, effective_from, effective_to, is_current
FROM workspace.freshmart_silver.scd_customers;

-- Dimension 2: Products
CREATE OR REPLACE TABLE workspace.freshmart_gold.dim_product AS
SELECT DISTINCT product_id, product_name, category
FROM workspace.freshmart_silver.clean_sales;

-- Dimension 3: Calendar Date Dimension
CREATE OR REPLACE TABLE workspace.freshmart_gold.dim_date AS
SELECT DISTINCT
  order_date AS date_key,
  YEAR(order_date) AS year,
  MONTH(order_date) AS month,
  DAY(order_date) AS day,
  DATE_FORMAT(order_date, 'EEEE') AS day_name
FROM workspace.freshmart_silver.clean_sales;

-- Fact Table: Sales (Center of the Star Schema)
CREATE OR REPLACE TABLE workspace.freshmart_gold.fact_sales AS
SELECT
  s.order_id,
  s.order_date AS date_key,
  c.customer_sk,
  s.product_id,
  s.qty,
  s.unit_price,
  s.total_amount
FROM workspace.freshmart_silver.clean_sales s
JOIN workspace.freshmart_gold.dim_customer c
  ON s.customer_id = c.customer_id
 AND s.order_date >= c.effective_from
 AND (c.effective_to IS NULL OR s.order_date < c.effective_to);

-- Optimize Gold Fact Table for fast dashboard queries
OPTIMIZE workspace.freshmart_gold.fact_sales;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1795628086113, 1795628087137, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE HISTORY workspace.freshmart_gold.fact_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-25T17:34:43.905Z,74201419967964,ingle13agi@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""bf258bc4-9d88-42c3-8368-0e004ce226c8"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-91e5c531-f735-4a6c-8fb7-505f33fbba1d"",""delta.enableInCommitTimestamps"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-fb5445b8-d5f9-4e41-bb27-b1bb42f494d4""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(3700768789777262),3485c3ae-1bbd-4c86-9eb0-d0d370407ac1,1125-172004-cpc2wxxk-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 4, numOutputBytes -> 2238)",null,Databricks-Runtime/19.16.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT * FROM workspace.freshmart_gold.fact_sales VERSION AS OF 0;

order_id,date_key,customer_sk,product_id,qty,unit_price,total_amount
O106,2026-11-03,3,P01,1,500.0,500.0
O103,2026-11-02,3,P03,5,120.0,600.0
O101,2026-11-01,1,P01,2,500.0,1000.0
O102,2026-11-01,2,P02,3,250.0,750.0
